In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# การใช้งาน Workload

<span id="usage"></span>
Usage แสดงถึงการบริโภค Qiskit Runtime service และถูกกำหนดโดยปริมาณเวลาที่ QPU ถูกล็อกเพื่อรัน workloads

- Session usage วัดเป็นเวลาที่ผ่านไปขณะที่ session ยังคงทำงานอยู่ เนื่องจากความจุ QPU ถูกสำรองไว้ตลอดระยะเวลาของ session โดยไม่คำนึงถึงว่า workload กำลังรันอยู่หรือไม่ ดู [ความยาวของ Session](/guides/run-jobs-session#session-length) สำหรับข้อมูลเพิ่มเติมเกี่ยวกับการเปลี่ยนสถานะของ session
- Batch usage วัดเป็นเวลาสะสมที่ QPU ถูกล็อกเพื่อรัน job ทั้งหมดใน batch
- Single job usage วัดเป็นเวลาที่ QPU ถูกล็อกเพื่อรัน job

โปรดทราบว่า job ที่ล้มเหลวหรือถูกยกเลิกนับรวมใน usage ของคุณในบางกรณี ดูส่วน [Job ที่ล้มเหลวและถูกยกเลิก](#failed-job) สำหรับรายละเอียด

สำหรับผู้ใช้ Pay-As-You-Go Plan ดู [จัดการค่าใช้จ่าย](/guides/manage-cost) สำหรับรายละเอียดเกี่ยวกับการตั้งค่าขีดจำกัดค่าใช้จ่าย

<span id="failed-job"></span>
## Usage สำหรับ job ที่ล้มเหลวและถูกยกเลิก
เมื่อ job ล้มเหลวหรือถูกยกเลิก usage ที่รายงานเป็นดังนี้:

* โหมด Job หรือ Batch: หากการล้มเหลวหรือการยกเลิกเกิดขึ้นเนื่องจากข้อผิดพลาดของระบบ usage ที่รายงานจะเป็นศูนย์ สำหรับ job ที่ล้มเหลวเนื่องจากข้อผิดพลาดของผู้ใช้หรือเมื่อผู้ใช้ยกเลิก job usage ที่รายงานคือปริมาณการบริโภคที่เกิดขึ้นจนถึงจุดนั้น รวมถึง overhead ที่เกิดขึ้นในการเตรียม QPU เพื่อรัน job

* โหมด Session: usage ที่รายงานคือเวลาตาม wall-clock ที่ session ทำงาน โดยไม่คำนึงถึงจำนวน job ที่ล้มเหลวหรือถูกยกเลิก

<span id="view-usage"></span>
## ดู usage จริงของ workload
หลังจาก workload เสร็จสมบูรณ์ มีหลายวิธีในการดู usage จริง:

- รัน [`batch.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/batch#usage) หรือ [`session.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session#usage) ใน `qiskit-ibm-runtime` 0.30 หรือใหม่กว่า หากใช้เวอร์ชันเก่ากว่าของ `qiskit-ibm-runtime` (>= 0.23 และ < 0.30) ยังสามารถพบ usage ได้ใน `session.details()["usage_time"]` และ `batch.details()["usage_time"]`
- ใช้ [`GET /sessions/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/sessions#tags__sessions__operations__GetSessionDetailsExtendedController_getSessionDetails) เพื่อดู usage สำหรับ batch หรือ session เฉพาะ
- ใช้ [`GET /jobs/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/jobs#tags__jobs__operations__GetJobByIdController_getJobById) เพื่อดู usage สำหรับ job เดียว

<span id="instance-usage"></span>
## ดู usage ของ instance
สามารถดู usage ของ instance ได้ที่หน้า [Instances](https://quantum.cloud.ibm.com/instances) หรือสำหรับผู้ที่มีสิทธิ์ที่เหมาะสม ที่หน้า [Analytics](https://quantum.cloud.ibm.com/analytics) โปรดทราบว่าหน้าต่างๆ อาจแสดงตัวเลข usage ที่แตกต่างกันเพราะคำนวณ usage ต่างกัน

หน้า Instances แสดง usage แบบ real-time สำหรับ 28 วันที่ผ่านมา (แบบ rolling) จนถึงเวลาปัจจุบันของวันนี้ Usage ในหน้า Analytics จะถูกคำนวณใหม่ทุกชั่วโมงและรวม 28 วันเต็มที่ผ่านมา นั่นคือ แสดง usage ตั้งแต่ 00:00 เมื่อ 28 วันก่อนจนถึงวันนี้ ณ ต้นชั่วโมง

## ประมาณ usage ก่อนส่ง job
แม้ว่าการประมาณแบบ local ที่แม่นยำจะซับซ้อนเนื่องจากการดำเนินการเพิ่มเติมสำหรับการกดและลดข้อผิดพลาด คุณสามารถใช้สูตรพื้นฐานนี้เพื่อประมาณ usage โดยประมาณ:

`<per sub-job overhead> + (rep_delay + <circuit length>) * <num executions>`

- `<per sub-job overhead>` คือ overhead ประมาณ 2 วินาทีต่อ sub-job ซึ่งรวมถึงการดำเนินการเช่นการโหลด payload เข้าสู่ control electronics primitive job ของคุณอาจถูกแบ่งออกเป็นหลาย sub-jobs หากมันใหญ่เกินไปสำหรับ execution engine ในการประมวลผลทั้งหมดในครั้งเดียว
- `rep_delay` เป็นตัวเลือก[ที่ผู้ใช้ปรับแต่งได้](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2#rep_delay) และค่าเริ่มต้นถูกกำหนดโดย `backend.default_rep_delay` ซึ่งคือ 250 microseconds บน IBM Quantum backends ส่วนใหญ่ โปรดทราบว่าการลด `rep_delay` ลดเวลารัน QPU ทั้งหมด แต่แลกกับการเพิ่มอัตราข้อผิดพลาดในการเตรียมสถานะ ดูคู่มือ [Dynamic repetition rate execution](/guides/repetition-rate-execution) สำหรับข้อมูลเพิ่มเติม
- `<circuit length>` คือความยาวของคำสั่งทั้งหมด แต่ละคำสั่งใช้เวลาต่างกันบน QPU ดังนั้นความยาวทั้งหมดจะแตกต่างกันไปตาม Circuit การวัดตัวอย่างเช่น อาจใช้เวลานานกว่า `x` gate ถึง 56 เท่า `backend.target[<instruction>][<qubit>].duration` สามารถใช้หาระยะเวลาที่แน่นอนของแต่ละคำสั่ง ความยาว Circuit ทั่วไปน่าจะอยู่ระหว่าง 50-100 microseconds หากคุณใช้เทคนิคการกดหรือลดข้อผิดพลาดกับ primitives อาจมีการแทรกคำสั่งเพิ่มเติมใน Circuit ของคุณ ซึ่งจะเพิ่มความยาว Circuit ทั้งหมด
    > **Note:** [ตัวเลือก `scheduler_timing` แบบทดลอง](/guides/visualize-circuit-timing) ส่งคืนเวลา Circuit ทั้งหมด แต่นี่ไม่ใช่เวลาที่ใช้สำหรับการเรียกเก็บเงิน
- `<num executions>` คือจำนวน Circuit ทั้งหมดคูณจำนวน shots โดยที่ Circuit คือที่สร้างหลังจาก PUB elements ถูก broadcast
    - หากคุณใช้เทคนิคการลดข้อผิดพลาดกับ primitives อาจมีการรัน Circuit เพิ่มเติมเป็นส่วนหนึ่งของกระบวนการลดข้อผิดพลาด ซึ่งจะเพิ่มจำนวนการรันทั้งหมด นอกจากนี้ เทคนิคการลดข้อผิดพลาดขั้นสูงเช่น PEA และ PEC มี overhead ที่สูงมากเพราะต้องรัน Circuit สำหรับ noise learning
    - Estimator จัดกลุ่ม observable ที่สลับกันแบบ qubit-wise ซึ่งช่วยลดจำนวนการรัน

หากคุณไม่ได้ใช้เทคนิคการลดข้อผิดพลาดขั้นสูงหรือ `rep_delay` แบบกำหนดเอง สามารถใช้ `2+0.00035*<num executions>` เป็นสูตรเร็วได้

### ประมาณ usage ในเครื่องด้วย Qiskit
ตัวอย่างโค้ดนี้แสดงวิธีใช้ Qiskit ในการคำนวณเวลา circuit: